In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import json
import glob
import requests

In [2]:
def setup_datasets():
    # 1. Clone AVeriTeC (Fact-checking Claims—used by Claimify)
    if not os.path.exists("averitec"):
        print("Cloning AVeriTeC...")
        subprocess.run(["git", "clone", "https://github.com/MichSchli/AVeriTeC.git", "averitec"])

    # 2. Clone SemEval-2020 Task 11 (Span-level Propaganda Labels)
    if not os.path.exists("semeval2020"):
        print("Downloading SemEval-2020 Task 11 dataset...")
        !wget -O "{"semeval-dataset.tgz"}" "{"https://zenodo.org/record/3952415/files/datasets-v2.tgz?download=1"}"
        print("Download complete.")

    # 3. Clone HQP (High-Quality Propaganda Social Media Dataset)
    if not os.path.exists("hqp"):
        print("Cloning HQP Dataset Repo...")
        subprocess.run(["git", "clone", "https://github.com/abdumaa/HiQualProp.git", "hqp"])

    print("\nDatasets initialized.")

if __name__ == "__main__":
    setup_datasets()

Cloning AVeriTeC...
--2026-02-14 01:08:18--  https://zenodo.org/record/3952415/files/datasets-v2.tgz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/3952415/files/datasets-v2.tgz [following]
--2026-02-14 01:08:18--  https://zenodo.org/records/3952415/files/datasets-v2.tgz
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1142202 (1.1M) [application/octet-stream]
Saving to: ‘semeval-dataset.tgz’

semeval-dataset.tgz 100%[===================>]   1.09M  1.42MB/s    in 0.8s    

2026-02-14 01:08:19 (1.42 MB/s) - ‘semeval-dataset.tgz’ saved [1142202/1142202]

Download complete.
Cloning HQP Dataset Repo...

Datasets initialized.


In [3]:
#Load AVeriTeC data into a dataframe
def load_averitec(file_path='averitec/data/dev.json'):
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Flattening the JSON structure
    df = pd.json_normalize(data)
    return df

averitec = load_averitec()
averitec.head()

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,None,None,https://web.archive.org/web/20201130144023/htt...,Facebook,None,[Event/Property Claim],[Written Evidence],[{'question': 'Where was the claim first publi...,None
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,None,None,https://web.archive.org/web/20201103001419/htt...,Instagram,US,"[Position Statement, Event/Property Claim]",[Written Evidence],[{'question': 'Has the Trump administration vo...,None
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,"[Causal Claim, Event/Property Claim]",[Written Evidence],[{'question': 'How did Macron criticise Islam?...,https://web.archive.org/web/20201113115127/htt...
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,None,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,[Event/Property Claim],[Written Evidence],"[{'question': 'What is Nadar?', 'answers': [{'...",None
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,,None,https://web.archive.org/web/20210713185816/htt...,Facebook,US,"[Numerical Claim, Event/Property Claim]","[Written Evidence, Numerical Comparison]",[{'question': 'Did Matt Gaetz work for Chemed ...,None


In [4]:
#Load HQP data into a dataframe
def load_hqp_labels(file_path='/content/hqp/Data/HiQualProp/HiQualProp.csv'):
    # This loads the IDs and the 'propaganda_strategy' labels
    df = pd.read_csv(file_path)
    return df

#id column shows Tweet ID but we have to run with Twitter API to get actual Tweet text
hqp = load_hqp_labels()
hqp.head()

,id,labels,strategy,labels_weak1,labels_weak3
0,1372969669389393932,0,general discourse,0,0
1,1521594471279996928,0,general discourse,0,0
2,1385896329071632386,0,general discourse,0,1
3,1403386698835365888,0,general discourse,0,1
4,1509240723362795531,0,general discourse,0,1


In [11]:
tgz_path = "semeval-dataset.tgz"
print(os.path.exists(tgz_path), tgz_path)

# Make a target directory
extract_dir = "semeval2020_data"
os.makedirs(extract_dir, exist_ok=True)

# Extract with tar
!tar -xvzf "$tgz_path" -C "$extract_dir"

True semeval-dataset.tgz
datasets/
datasets/README.md
datasets/train-labels-task1-span-identification/
datasets/train-labels-task1-span-identification/article728972961.task1-SI.labels
datasets/train-labels-task1-span-identification/article111111136.task1-SI.labels
datasets/train-labels-task1-span-identification/article790667730.task1-SI.labels
datasets/train-labels-task1-span-identification/article770156851.task1-SI.labels
datasets/train-labels-task1-span-identification/article786250729.task1-SI.labels
datasets/train-labels-task1-span-identification/article766632016.task1-SI.labels
datasets/train-labels-task1-span-identification/article786344683.task1-SI.labels
datasets/train-labels-task1-span-identification/article999000880.task1-SI.labels
datasets/train-labels-task1-span-identification/article999000147.task1-SI.labels
datasets/train-labels-task1-span-identification/article696694316.task1-SI.labels
datasets/train-labels-task1-span-identification/article781847297.task1-SI.labels
datase

In [6]:
base_dir = "semeval2020_data"

si_files = []
tc_files = []

for path in glob.glob(os.path.join(base_dir, "**", "*.labels"), recursive=True):
    fname = os.path.basename(path)
    if "task1-SI" in fname:
        si_files.append(path)
    elif "task2-TC" in fname:
        tc_files.append(path)

# Load SI (no technique column)
df_si_list = []
for path in si_files:
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["article_id", "start_char", "end_char"]
    )
    df["source_file"] = os.path.basename(path)
    df_si_list.append(df)
df_si = pd.concat(df_si_list, ignore_index=True)

# Load TC (with technique column second)
df_tc_list = []
for path in tc_files:
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["article_id", "technique", "start_char", "end_char"]
    )
    df["source_file"] = os.path.basename(path)
    df_tc_list.append(df)
df_tc = pd.concat(df_tc_list, ignore_index=True)
df_tc

,article_id,technique,start_char,end_char,source_file
0,111111111,Appeal_to_Authority,265,323,train-task2-TC.labels
1,111111111,Appeal_to_Authority,1795,1935,train-task2-TC.labels
2,111111111,Doubt,149,157,train-task2-TC.labels
3,111111111,Repetition,1069,1091,train-task2-TC.labels
4,111111111,Appeal_to_fear-prejudice,1334,1462,train-task2-TC.labels
...,...,...,...,...,...
12253,730573740,Black-and-White_Fallacy,4118,4224,article730573740.task2-TC.labels
12254,730573740,"Name_Calling,Labeling",1302,1322,article730573740.task2-TC.labels
12255,730573740,"Name_Calling,Labeling",1445,1467,article730573740.task2-TC.labels
12256,999000880,Flag-Waving,3114,3143,article999000880.task2-TC.labels


In [7]:
text_files = glob.glob(os.path.join(base_dir, "**", "*.txt"), recursive=True)
print("Found text files:", text_files[:10])  # show a sample

articles = []

for path in text_files:
    fname = os.path.basename(path)          # e.g. "article763280007.txt"
    article_id_str = os.path.splitext(fname)[0]  # "article763280007"
    # Strip the "article" prefix if present
    if article_id_str.startswith("article"):
        article_id_str = article_id_str[len("article"):]  # "763280007"
    article_id = int(article_id_str)          # numeric to match labels

    with open(path, encoding="utf-8") as f:
        text = f.read()

    articles.append({"article_id": article_id, "text": text})

df_articles = pd.DataFrame(articles)
df_articles.head()

Found text files: ['semeval2020_data/datasets/dev-articles/article763280007.txt', 'semeval2020_data/datasets/dev-articles/article782448403.txt', 'semeval2020_data/datasets/dev-articles/article832910505.txt', 'semeval2020_data/datasets/dev-articles/article999000858.txt', 'semeval2020_data/datasets/dev-articles/article779309765.txt', 'semeval2020_data/datasets/dev-articles/article787085939.txt', 'semeval2020_data/datasets/dev-articles/article787730392.txt', 'semeval2020_data/datasets/dev-articles/article776385494.txt', 'semeval2020_data/datasets/dev-articles/article776126299.txt', 'semeval2020_data/datasets/dev-articles/article740235127.txt']


,article_id,text
0,763280007,Geert Wilders Puts the Political Elites On Not...
1,782448403,Federal judge rules against Trump administrati...
2,832910505,"The Mueller Circus is Over, Long Live the Muel..."
3,999000858,Stacey Abrams to Sue Georgia Over ‘Gross Misma...
4,779309765,Unbelievable!\nSharia New Mexico: Islamic comp...


In [8]:
sem_eval = df_tc.merge(df_articles[["article_id", "text"]], on="article_id", how="left")
sem_eval.head()

,article_id,technique,start_char,end_char,source_file,text
0,111111111,Appeal_to_Authority,265,323,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
1,111111111,Appeal_to_Authority,1795,1935,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
2,111111111,Doubt,149,157,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
3,111111111,Repetition,1069,1091,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
4,111111111,Appeal_to_fear-prejudice,1334,1462,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...


In [9]:
# Convert comma-separated technique strings to lists
sem_eval['technique'] = sem_eval['technique'].str.split(',')

# Clean up whitespace from each technique name
sem_eval['technique'] = sem_eval['technique'].apply(lambda x: [t.strip() for t in x])
sem_eval

,article_id,technique,start_char,end_char,source_file,text
0,111111111,[Appeal_to_Authority],265,323,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
1,111111111,[Appeal_to_Authority],1795,1935,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
2,111111111,[Doubt],149,157,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
3,111111111,[Repetition],1069,1091,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
4,111111111,[Appeal_to_fear-prejudice],1334,1462,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...
...,...,...,...,...,...,...
12253,730573740,[Black-and-White_Fallacy],4118,4224,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...
12254,730573740,"[Name_Calling, Labeling]",1302,1322,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...
12255,730573740,"[Name_Calling, Labeling]",1445,1467,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...
12256,999000880,[Flag-Waving],3114,3143,article999000880.task2-TC.labels,They Are Coming: Migrant Caravan Resumes March...


In [10]:
#Only show relevant text being evaluated for each row (using span on text)
sem_eval['span_text'] = sem_eval.apply(lambda row: row['text'][row['start_char']:row['end_char']], axis=1)
sem_eval

,article_id,technique,start_char,end_char,source_file,text,span_text
0,111111111,[Appeal_to_Authority],265,323,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...
1,111111111,[Appeal_to_Authority],1795,1935,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...,when (the plague) comes again it starts from m...
2,111111111,[Doubt],149,157,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...,appeared
3,111111111,[Repetition],1069,1091,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...,"a very, very different"
4,111111111,[Appeal_to_fear-prejudice],1334,1462,train-task2-TC.labels,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...
...,...,...,...,...,...,...,...
12253,730573740,[Black-and-White_Fallacy],4118,4224,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...,The Commandments are an expression of the Love...
12254,730573740,"[Name_Calling, Labeling]",1302,1322,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...,theocentric humanism
12255,730573740,"[Name_Calling, Labeling]",1445,1467,article730573740.task2-TC.labels,Cardinal Müller: Blessing homosexual couples w...,suicidal modernization
12256,999000880,[Flag-Waving],3114,3143,article999000880.task2-TC.labels,They Are Coming: Migrant Caravan Resumes March...,We need people in our country
